# 第８回　局所的な位置合わせ（warping）（医療画像処理工学演習）

このノートブックでは，場所によってずれ方が異なる過去画像と現在画像について，
**画像全体を1回だけ平行移動する方法**と，**ブロックごとにずれ量を求めて合わせる方法**，
さらに**ずれ量をなめらかにつないで画像を変形する方法（warping）**を比べます．

**使い方**：上のセルから順に，セルの左にある ▶ を押してください．コードを書き換える必要はありません．
各セルの実行が終わると，▶ の左に緑色のチェックが付きます．

**うまくいかないとき**：メニューの「ランタイム」→「ランタイムを接続解除して削除」を選び，１．からやり直してください．

---
使用する画像：胸部X線画像に，模擬の陰影と，呼吸を想定した変形（場所によってずれ方が異なる）を加えたもの．512×512画素，画素サイズ0.7 mm．

In [ ]:
#@title １．過去画像と現在画像を読み込む
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import map_coordinates
from scipy.interpolate import RegularGridInterpolator

BASE = 'https://github.com/ichikawalab/practice-medical-imaging/raw/main/imgproc/08_subtraction/data/'
for name in ('warp_past.tif', 'warp_current.tif'):
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + name, name)
past = np.array(Image.open('warp_past.tif'), dtype=float)
cur = np.array(Image.open('warp_current.tif'), dtype=float)
H, W = cur.shape

MARGIN = 16     # 画像の端（平行移動で値がなくなる部分）を評価から除く幅［画素］
SEARCH = 12     # ずれ量を探す範囲：-12〜+12画素
WIN = 2000      # 差分画像の表示：Level 0，Window 4000（-2000〜+2000）
yy, xx = np.mgrid[0:H, 0:W].astype(float)

def translate(img, dx, dy):
    # ImageJの Translate（Interpolation: None）と同じ：画像を (dx, dy) 画素ずらし，はみ出た部分は0
    out = np.zeros_like(img)
    out[max(dy, 0):H + min(dy, 0), max(dx, 0):W + min(dx, 0)] = \
        img[max(-dy, 0):H + min(-dy, 0), max(-dx, 0):W + min(-dx, 0)]
    return out

def sd(diff):
    # 差分画像の標準偏差（画像の端を除く）．小さいほど位置が合っている
    return diff[MARGIN:-MARGIN, MARGIN:-MARGIN].std()

def show_diff(ax, diff, title):
    ax.imshow(diff, cmap='gray', vmin=-WIN, vmax=WIN)
    ax.set_title(title); ax.axis('off')

fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
for a, img, t in ((ax[0], past, 'Past'), (ax[1], cur, 'Current')):   # グラフの文字は英語で表示
    a.imshow(img, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()
print('左：過去画像，右：現在画像．')
print('できました．２．に進んでください．')

In [ ]:
#@title ２．画像全体を1回だけ平行移動して合わせる
# 演習8-1と同じ方法：ずれ量を -12〜+12 画素の範囲ですべて試し，標準偏差が最も小さいものを選ぶ
diff0 = cur - past
best = min(((dx, dy) for dx in range(-SEARCH, SEARCH + 1) for dy in range(-SEARCH, SEARCH + 1)),
           key=lambda d: sd(cur - translate(past, *d)))
diff_global = cur - translate(past, *best)

print('標準偏差は，画像の端（16画素）を除いた範囲で計算しています．')
print(f'位置合わせなし　　　　　　　：標準偏差 {sd(diff0):7.0f}')
print(f'全体を平行移動 X={best[0]:+d}, Y={best[1]:+d}：標準偏差 {sd(diff_global):7.0f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
show_diff(ax[0], diff0, 'No registration')
show_diff(ax[1], diff_global, f'Global shift (X={best[0]:+d}, Y={best[1]:+d})')
plt.show()
print('左：位置合わせなしの差分，右：全体を平行移動した差分（Level 0，Window 4000）．')
print('できました．３．に進んでください．')

In [ ]:
#@title ３．ブロックごとにずれ量を求めて合わせる（ブロック数を選んでから ▶ を押す）
ブロック数 = '2×2'  #@param ['2×2', '8×8']
n = int(ブロック数[0])
bs = H // n

# すべてのずれ量について差分をとり，ブロックごとに標準偏差が最も小さいずれ量を選ぶ
inner = np.zeros((H, W), bool); inner[MARGIN:-MARGIN, MARGIN:-MARGIN] = True
best_sd = np.full((n, n), np.inf); shift = np.zeros((n, n, 2), int)
for dx in range(-SEARCH, SEARCH + 1):
    for dy in range(-SEARCH, SEARCH + 1):
        d = cur - translate(past, dx, dy)
        for i in range(n):
            for j in range(n):
                blk = (slice(i * bs, (i + 1) * bs), slice(j * bs, (j + 1) * bs))
                s = d[blk][inner[blk]].std()
                if s < best_sd[i, j]:
                    best_sd[i, j] = s; shift[i, j] = dx, dy

# ブロックごとに平行移動した差分画像をつなぎ合わせる（ImageJの Combine に相当）
diff_block = np.zeros((H, W))
for i in range(n):
    for j in range(n):
        blk = (slice(i * bs, (i + 1) * bs), slice(j * bs, (j + 1) * bs))
        diff_block[blk] = (cur - translate(past, *shift[i, j]))[blk]

print(f'ブロック数 {ブロック数}：標準偏差 {sd(diff_block):.0f}')
print('各ブロックのずれ量（X, Y）［画素］：')
for i in range(n):
    print('  ' + '  '.join(f'({shift[i, j, 0]:+d},{shift[i, j, 1]:+d})' for j in range(n)))

cy, cx = (np.arange(n) + 0.5) * bs, (np.arange(n) + 0.5) * bs
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(cur, cmap='gray')
ax[0].quiver(*np.meshgrid(cx, cy), shift[..., 0], shift[..., 1], color='orange',
             angles='xy', scale_units='xy', scale=0.15, width=0.006)
for k in range(1, n):
    ax[0].axhline(k * bs, color='c', lw=0.8); ax[0].axvline(k * bs, color='c', lw=0.8)
ax[0].set_title(f'Shift of each block ({n}x{n})'); ax[0].axis('off')
show_diff(ax[1], diff_block, f'Block-wise shift ({n}x{n})')
plt.show()
print('左：各ブロックのずれ（オレンジの矢印，長さは約7倍に拡大），右：ブロックごとに平行移動した差分．')
print('できました．４．に進んでください．')

In [ ]:
#@title ４．ずれ量をなめらかにつないで画像を変形する（warping）
# ３．で求めた各ブロック中心のずれ量を，双線形補間で画素ごとにつないで「ずれの地図」を作り，過去画像を変形する
f = lambda v: RegularGridInterpolator((cy, cx), v, bounds_error=False, fill_value=None)
pts = np.stack([np.clip(yy, cy[0], cy[-1]), np.clip(xx, cx[0], cx[-1])], -1)
ux, uy = (f(shift[..., 0].astype(float))(pts), f(shift[..., 1].astype(float))(pts)) if n > 1 else \
         (np.full((H, W), float(shift[0, 0, 0])), np.full((H, W), float(shift[0, 0, 1])))
warped = map_coordinates(past, [yy - uy, xx - ux], order=1, cval=0)
diff_warp = cur - warped

print(f'warping（{ブロック数}のずれ量から）：標準偏差 {sd(diff_warp):.0f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(warped, cmap='gray')
g = np.arange(0, H + 1, 32)
for v in g:   # 方眼紙をどのように変形したかを格子で示す（ずれは約3倍に拡大）
    ax[0].plot(np.clip(v + 3 * ux[np.clip(v, 0, H - 1), :], 0, W - 1), np.arange(W), 'y', lw=0.6)
    ax[0].plot(np.arange(H), np.clip(v + 3 * uy[:, np.clip(v, 0, W - 1)], 0, H - 1), 'y', lw=0.6)
ax[0].set_title('Warped past image and deformation grid'); ax[0].axis('off')
show_diff(ax[1], diff_warp, f'Warping (from {n}x{n} shifts)')
plt.show()
print('ブロック中心のずれ量を双線形補間でなめらかにつなぎ，画素ごとのずれ量で過去画像を変形しました．')
print('左：変形した過去画像と，方眼をどう変形したかを示す黄色の格子（ずれは約3倍に拡大），右：warpingした差分．')
print('できました．５．に進んでください．')

In [ ]:
#@title ５．結果を比べる
rows = [('位置合わせなし', 'No registration', diff0), ('全体を平行移動', 'Global shift', diff_global),
        (f'ブロックごとに平行移動（{ブロック数}）', f'Block-wise ({n}x{n})', diff_block),
        (f'warping（{ブロック数}）', f'Warping ({n}x{n})', diff_warp)]
print('標準偏差（画像の端16画素を除いた範囲で計算）：')
for t, _, d in rows:
    print(f'{t:　<20}標準偏差 {sd(d):7.0f}')
fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
for a, (_, e, d) in zip(ax, rows):
    show_diff(a, d, e)
plt.show()
print('左から，位置合わせなし，全体を平行移動，ブロックごとに平行移動，warping の差分．')
if n == 2:
    print('できました．３．に戻ってブロック数を「8×8」に変え，３．から５．をもう一度実行してください．')
else:
    print('できました．')